# Етап 3.1–3.2: довідник "порт → сервіс → продукт" і функція CVE-пулу
Будує:
1. `PORT_SERVICE_MAP` — порт/протокол → назва сервісу → кандидати `vendor:product` для пошуку в `cve_master`.
2. `port_to_service()` — розбір порту з явною гілкою для tcp/udp, icmp, arp.
3. `build_service_pool()` — попередньо рахує для кожного сервісу відфільтрований і відсортований за EPSS пул CVE (кешується один раз, а не на кожен рядок). Токен `'windows'` НЕ фільтрується жорстко — замість цього повертаються діагностичні `n_generic_os_matches`/`n_specific_product_matches`.
4. `attach_cve_context()` — приєднує до датасету колонки контексту (`cve_ctx_service`, `cve_ctx_pool_size`, `cve_ctx_service_max_epss`, `cve_ctx_service_in_kev`, `cve_ctx_use_epss` тощо) через join по (порт, протокол), а не по рядку — швидко навіть на мільйони потоків.
5. Sanity-check на 8 типових сервісах (http/https/ssh/ftp/smb/rdp/quic/rpcbind) ПЕРЕД застосуванням до повних датасетів.
6. Застосування до UNSW-NB15, CIC-IDS2017, GeNIS + діагностика **асоціації** CVE-контексту з міткою атаки.

**Важливе термінологічне уточнення:** діагностика в кроці 6 показує, що значення контексту
відрізняються між класами атак ("CVE-context / label association check") — це очікуваний і
бажаний сигнал (саме на нього розрахована гіпотеза статті), а НЕ доказ витоку мітки. Захист
від часового витоку забезпечує окрема, вже коректна конструкція: `published <= snapshot_date`,
`kev_date_added <= snapshot_date`, вибір EPSS-знімка не пізнішого за snapshot_date.

**Про `cve_ctx_service_max_epss` / `cve_ctx_service_in_kev`:** значення обчислені на рівні
СЕРВІСУ/ПОРТУ (усі потоки з однаковим портом отримують однакове значення) — це "port/service-
associated CVE exposure", а не факт про конкретний flow чи хост. Через це контекст диференціює
МІЖ сервісами, але не ВСЕРЕДИНІ одного сервісу — обмеження, яке варто явно визнати в статті.

**Про EPSS для історичних дат (UNSW-NB15 2015, CIC-IDS2017 2017):** EPSS не публікувався до
2021-04-14, тому `cve_ctx_epss_is_retrospective=True` для обох, а `cve_ctx_use_epss=False` —
EPSS для цих датасетів залишається контекстною діагностикою, ніколи не часовою ознакою в ML.

Ядро: IDS Research.


In [1]:
import re, bisect, gzip
from datetime import date
from pathlib import Path
import pandas as pd
pd.set_option("display.max_colwidth", 60, "display.width", 200)

CTI_DIR = Path("/srv/ids_research/data/cti")
master = pd.read_parquet(CTI_DIR / "cve_master.parquet")
master["published"] = pd.to_datetime(master["published"], errors="coerce")
master["kev_date_added"] = pd.to_datetime(master.get("kev_date_added"), errors="coerce")
print("cve_master:", master.shape, "| з датою KEV:", master["kev_date_added"].notna().sum())

# ---------- вибір знімка EPSS за датою ----------
snaps = sorted((CTI_DIR / "epss" / "history").glob("epss_scores-*.csv.gz"))
snap_dates = [date.fromisoformat(p.name[12:22]) for p in snaps]

def first_line_gz(p):
    with gzip.open(p, "rt", errors="replace") as f:
        return f.readline().strip()

_epss_cache = {}
def epss_at(d):
    """Найближчий знімок EPSS НЕ ПІЗНІШЕ за дату d.
    epss_is_retrospective=True означає: дата d раніша за перший доступний знімок EPSS
    (EPSS почав публікуватися 2021-04-14), і тому взято найраніший наявний знімок,
    приписаний ЗАДНІМ ЧИСЛОМ CVE, опублікованим до 2021 року. Це НЕ "EPSS, яким він
    був на дату d" — такого значення просто не існує, EPSS тоді ще не публікувався.
    Використовувати таке значення як часову ознаку в leakage-чутливому експерименті нельзя."""
    i = bisect.bisect_right(snap_dates, d) - 1
    epss_is_retrospective = i < 0
    if i < 0:
        i = 0
    if i not in _epss_cache:
        # РІЗНІ знімки EPSS мають РІЗНУ структуру колонок: найраніші файли (до v2022.01.01,
        # квітень 2021 — лютий 2022) публікувалися без колонки 'percentile' взагалі.
        # Пряме ["cve","epss","percentile"] падає з KeyError на таких файлах — розбираємо
        # стійко: беремо колонку з CVE (перша, або та, що зветься 'cve'), колонку 'epss'
        # (або першу, де в назві є 'epss'), а 'percentile' — якщо є, інакше NaN.
        raw = pd.read_csv(snaps[i], comment="#")
        raw.columns = [str(c).strip().lower() for c in raw.columns]
        cve_col = "cve" if "cve" in raw.columns else raw.columns[0]
        epss_col = "epss" if "epss" in raw.columns else next(
            (c for c in raw.columns if "epss" in c), raw.columns[1])
        df = raw.rename(columns={cve_col: "cve_id", epss_col: "epss"})
        if "percentile" not in df.columns:
            df["percentile"] = pd.NA
            print(f"  ⚠ {snaps[i].name}: немає колонки 'percentile' у цьому знімку EPSS "
                  f"(типово для найраніших файлів, до v2022.01.01) — заповнено NaN")
        df = df[["cve_id", "epss", "percentile"]]
        ver = re.search(r"model_version:(v[\d.]+)", first_line_gz(snaps[i]))
        _epss_cache[i] = (df, snap_dates[i], ver.group(1) if ver else "v1?")
    df, used_date, ver = _epss_cache[i]
    return df, used_date, ver, epss_is_retrospective

cve_master: (395739, 17) | з датою KEV: 1716


In [2]:
import itertools
# ---------- 1. ПОРТ (+ПРОТОКОЛ) → СЕРВІС → ТОЧНІ ТОКЕНИ ПРОДУКТУ ----------
# Токени звіряються з ОСТАННІМ сегментом рядка "vendor:product" у колонці cve_master["products"]
# (точний збіг, а не substring-пошук) — це усуває хибні збіги на кшталт "php" всередині
# випадкового ідентифікатора. Токен також звіряється з повним рядком "vendor:product" для
# випадків, коли він заданий саме так (напр. "microsoft:windows").
# Це все одно ЕВРИСТИКА "порт → типовий клас ПЗ для такого сервісу", а не підтверджене
# ПЗ конкретного хоста: жоден з трьох датасетів не документує версію ПЗ жертви. У статті
# результат слід називати "CVE-пул, асоційований з портом/сервісом", а не
# "вразливості хоста".
#
# Для портів, де протокол змінює саму природу сервісу (443/tcp = HTTPS, але 443/udp дедалі
# частіше QUIC/HTTP-3), запис — словник {proto: (service, tokens)} замість кортежу.
PORT_SERVICE_MAP = {
    20:  ("ftp-data", ["vsftpd", "proftpd", "filezilla", "pure-ftpd", "iis"]),
    21:  ("ftp",      ["vsftpd", "proftpd", "filezilla", "pure-ftpd", "iis"]),
    22:  ("ssh",      ["openssh", "dropbear"]),
    23:  ("telnet",   ["telnetd", "busybox"]),
    25:  ("smtp",     ["postfix", "exim", "sendmail", "exchange_server"]),
    53:  ("dns",      ["bind", "dnsmasq", "powerdns", "unbound", "dns"]),
    80:  ("http",     ["http_server", "nginx", "iis", "php", "node.js"]),
    110: ("pop3",     ["dovecot", "courier", "exchange_server"]),
    111: ("rpcbind",  ["rpcbind"]),
    123: ("ntp",      ["ntp", "chrony"]),
    135: ("msrpc",    ["windows"]),
    137: ("netbios",  ["samba", "windows"]),
    138: ("netbios",  ["samba", "windows"]),
    139: ("smb",      ["samba", "windows"]),
    143: ("imap",     ["dovecot", "courier", "exchange_server"]),
    161: ("snmp",     ["net-snmp", "windows"]),
    389: ("ldap",     ["openldap", "active_directory"]),
    443: {"tcp": ("https", ["http_server", "nginx", "iis", "openssl"]),
          # QUIC: 'quic'/'msquic' НЕ трапляються як реальні vendor:product у корпусі
          # (перевірено окремим запитом до master['products']). Назва мережевого протоколу
          # ≠ назва продукту в CVE/CPE. Замість вигадування токена навмання, чесно
          # залишаємо це "без CVE-мапінгу": cve_ctx_context_available покаже False,
          # а не хибно прив'язаний до HTTP/Windows контекст.
          "udp": ("quic",  [])},
    445: ("smb",      ["samba", "windows"]),
    514: ("syslog",   ["rsyslog", "syslog-ng"]),
    587: ("smtp",     ["postfix", "exim", "sendmail", "exchange_server"]),
    993: ("imaps",    ["dovecot", "courier", "exchange_server"]),
    995: ("pop3s",    ["dovecot", "courier", "exchange_server"]),
    1433:("mssql",    ["sql_server"]),
    2049:("nfs",      ["linux_kernel", "nfs-utils"]),
    3306:("mysql",    ["mysql", "mariadb"]),
    3389:("rdp",      []),  # токени підставляються ДИНАМІЧНО нижче з реальних products корпусу
    5353:("mdns",     ["avahi", "bonjour"]),
    5432:("postgresql", ["postgresql"]),
    8080:("http-alt", ["tomcat", "http_server", "nginx"]),
}
print(f"Портів у довіднику: {len(PORT_SERVICE_MAP)}")

# ---- RDP: токени будуються З РЕАЛЬНИХ products корпусу, а не вгадуються ----
# 'remote_desktop_services' як довільний токен НЕ спрацював (0 vendor:product у корпусі
# містять такий рядок). Замість цього беремо ЛИШЕ офіційні Microsoft-компоненти RDP
# (vendor == microsoft, product починається з remote_desktop) — свідомо виключаючи
# сторонні "remote desktop"-інструменти інших вендорів (Apple Remote Desktop тощо),
# які не є тим самим протоколом на порту 3389.
_all_products = set(itertools.chain.from_iterable(
    ps for ps in master["products"] if ps is not None and len(ps) > 0))
_rdp_tokens = sorted({p.split(":", 1)[1] for p in _all_products if p.lower().startswith("microsoft:remote_desktop")})
PORT_SERVICE_MAP[3389] = ("rdp", _rdp_tokens)
print(f"RDP: {len(_rdp_tokens)} офіційних Microsoft RDP-токенів знайдено в корпусі: {_rdp_tokens}")



def parse_port(x):
    """Argus/CIC/UNSW-сумісний розбір порту: підтримує hex ('0x...') і числові рядки/float."""
    if pd.isna(x):
        return None
    s = str(x)
    try:
        if s.lower().startswith("0x"):
            return int(s, 16)
        return int(float(s))
    except (ValueError, TypeError):
        return None

# Порти, де прикладний сервіс за визначенням існує ЛИШЕ над TCP (HTTP, SSH, FTP, SMTP,
# IMAP/POP3(S), SMB, MySQL/MSSQL/PostgreSQL тощо). UDP-трафік на ці самі номери портів —
# майже завжди DoS-флуд/спуфінг проти "відомого" порту, а не легітимний UDP-сервіс, і
# НЕ повинен успадковувати CVE-пул TCP-сервісу (саме такий баг стався з портом 80: UDP-флуд
# на 80/udp хибно отримував service='http'). Свідомо НЕ включено сюди дуалі-протокольні
# сервіси (111 rpcbind, 123 ntp, 135 msrpc, 137/138 netbios, 161 snmp, 514 syslog, 2049 nfs,
# 3389 rdp), де UDP — теж легітимний, реальний транспорт для цього самого сервісу.
TCP_ONLY_PORTS = {20, 21, 22, 23, 25, 80, 110, 143, 389, 445, 587, 993, 995, 1433, 3306, 5432, 8080}

def port_to_service(port, proto):
    proto = str(proto).lower() if proto is not None else ""
    if proto == "arp":
        return "arp", []            # немає порту й сервісу в принципі
    if proto == "icmp":
        # ICMP визначає ПРОТОКОЛ, а не операційну систему чи застосунок — прив'язка до
        # конкретних vendor:product була б необґрунтованим узагальненням. Порожній
        # пул продуктів; за потреби пізніше можна додати окремий, явно позначений
        # "protocol-level" пул ICMP-специфічних CVE (не vendor/product-прив'язаних).
        return "icmp", []
    if port is None:
        return "unknown", []
    port = int(port)
    if port in TCP_ONLY_PORTS and proto != "tcp":
        # UDP-трафік на TCP-only порт — не успадковує CVE-пул TCP-сервісу
        return "other", []
    entry = PORT_SERVICE_MAP.get(port)
    if entry is None:
        return "other", []
    if isinstance(entry, dict):   # прото-залежний запис (напр. 443)
        return entry.get(proto, entry.get("tcp", ("other", [])))
    return entry

Портів у довіднику: 30
RDP: 7 офіційних Microsoft RDP-токенів знайдено в корпусі: ['remote_desktop', 'remote_desktop_app', 'remote_desktop_client', 'remote_desktop_connection', 'remote_desktop_connection_client', 'remote_desktop_connection_manager', 'remote_desktop_web_client']


In [3]:
# ---------- 2. ПОБУДОВА ПУЛУ CVE НА СЕРВІС (точний match токенів, KEV з датою, покриття EPSS) ----------
_service_pool_cache = {}

def _match_tokens(product_list, tokens):
    """Точний збіг: останній сегмент 'vendor:product' АБО весь рядок дорівнює одному з tokens.
    Це НЕ substring-пошук — усуває хибні збіги (напр. 'php' всередині іншого ідентифікатора).
    product_list може прийти з parquet як numpy-масив, а не Python list — 'if not product_list'
    на такому масиві падає з ValueError('truth value ... ambiguous'), тому перевіряємо через len()."""
    if product_list is None:
        return False
    try:
        if len(product_list) == 0:
            return False
    except TypeError:
        return False
    tok_set = set(tokens)
    for p in product_list:
        last = str(p).split(":", 1)[-1]
        if p in tok_set or last in tok_set:
            return True
    return False

def build_service_pool(service, product_tokens, snapshot_date, top_n=20):
    """
    ВАЖЛИВО ЩОДО ТРАКТУВАННЯ РЕЗУЛЬТАТУ:
    'max_epss' — це максимальний EPSS СЕРЕД CVE, АСОЦІЙОВАНИХ З ПОРТОМ/СЕРВІСОМ (евристично,
    через типові продукти цього класу сервісу). Це НЕ EPSS конкретної вразливості конкретного
    хоста — жоден з наших датасетів не документує точну версію ПЗ жертви. У статті результат
    слід називати "port/service-associated CVE exposure", а не "host has vulnerability X".

    Токен 'windows' навмисне НЕ фільтрується жорстко за описом (раніше тут стояв
    DESC_KEYWORD_FILTER, який мовчки виключав частину CVE із самого пулу — це ховало
    інформацію замість того, щоб її показувати). Замість фільтра результат несе ДІАГНОСТИЧНІ
    поля n_generic_os_matches / n_specific_product_matches: аналітик сам бачить, яка частка пулу спирається
    на широкий OS-маркер 'windows' (потенційно зашумлена), а яка — на специфічний для
    протоколу токен (samba, vsftpd, openssh тощо).
    """
    key = (service, tuple(product_tokens), snapshot_date, top_n)
    if key in _service_pool_cache:
        return _service_pool_cache[key]

    if not product_tokens:
        # Сервіс НАВМИСНЕ без товарного мапінгу (напр. QUIC) — не помилка, а чесна
        # відсутність надійного CPE-відповідника для цього мережевого протоколу.
        result = dict(service=service, n_cve=0, n_epss=0, epss_coverage=None, epss_available=False,
                      max_epss=None, in_kev=False, n_generic_os_matches=0, n_specific_product_matches=0,
                      top_cve=[], epss_snapshot_used=None, epss_model=None, epss_is_retrospective=None)
        _service_pool_cache[key] = result
        return result

    mask = master["products"].map(lambda ps: _match_tokens(ps, product_tokens))

    # ДІАГНОСТИКА (НЕ фільтр): скільки CVE в пулі потрапили ЛИШЕ через широкий токен 'windows'
    # (тобто не збіглися б за жодним іншим, специфічнішим токеном сервісу), а скільки —
    # через специфічний токен. Пул НЕ звужується — усі CVE лишаються, це лише прозорість.
    n_generic_os_matches, n_specific_product_matches = 0, int(mask.sum())
    if "windows" in product_tokens:
        non_windows_tokens = [t for t in product_tokens if t != "windows"]
        mask_specific = (master["products"].map(lambda ps: _match_tokens(ps, non_windows_tokens))
                          if non_windows_tokens else pd.Series(False, index=master.index))
        n_specific_product_matches = int((mask & mask_specific).sum())
        n_generic_os_matches = int((mask & ~mask_specific).sum())

    snap_ts = pd.Timestamp(snapshot_date)
    pool = master[mask & (master["published"].isna() | (master["published"] <= snap_ts))]

    # cve_master вже містить власну колонку 'epss' (актуальну на момент побудови корпусу) —
    # прибираємо її перед мержем ІСТОРИЧНОГО знімка, інакше pandas тихо перейменує обидві
    # в epss_x/epss_y і подальший код впаде на KeyError('epss') (та сама пастка, що й
    # з колонкою 'service' в UNSW-NB15).
    pool = pool.drop(columns=[c for c in ("epss", "epss_percentile") if c in pool.columns])

    epss_df, used_date, ver, epss_is_retro = epss_at(snapshot_date)
    pool = pool.merge(epss_df, on="cve_id", how="left")
    assert "epss" in pool.columns, "мерж історичного EPSS не додав колонку 'epss' — перевірте на конфлікт назв"
    pool = pool.sort_values("epss", ascending=False)

    # KEV — ЛИШЕ якщо дата включення в KEV сама не пізніша за snapshot_date (усуває
    # темпоральний витік: CVE не могла "бути в KEV" у 2015 році, якщо її додали в 2023-му)
    kev_valid = pool["kev_date_added"].notna() & (pool["kev_date_added"] <= snap_ts)

    n_cve = len(pool)
    n_epss = int(pool["epss"].notna().sum())
    top = pool.head(top_n)[["cve_id", "cvss_score", "epss", "kev_date_added"]].to_dict("records")
    result = dict(
        service=service,
        n_cve=n_cve,
        n_epss=n_epss,
        epss_coverage=(n_epss / n_cve) if n_cve else None,   # частка CVE в пулі, для яких EPSS реально є
        epss_available=n_epss > 0,           # чи є в пулі ХОЧА Б ОДНА CVE з EPSS-оцінкою
        max_epss=pool["epss"].max() if n_cve else None,
        in_kev=bool(kev_valid.any()) if n_cve else False,
        n_generic_os_matches=n_generic_os_matches,           # скільки в пулі потрапило ЛИШЕ через широкий 'windows'
        n_specific_product_matches=n_specific_product_matches,  # скільки підтверджено специфічним для сервісу токеном
        top_cve=top,
        epss_snapshot_used=used_date,
        epss_model=ver,
        epss_is_retrospective=epss_is_retro,  # True = дата раніша за перший знімок EPSS (2021-04-14);
                                               # значення epss/max_epss тоді ЗАДНІМ ЧИСЛОМ приписане
                                               # старим CVE і НЕ придатне як часова ознака в ML
    )
    if n_cve == 0:
        # 0 тут може бути ЛЕГІТИМНИМ (жодна відповідна CVE не опублікована до snapshot_date —
        # темпоральний фільтр коректно спрацював, як для rpcbind на даті CIC-IDS2017 2017-07-03,
        # де ті самі токени дають 4 CVE на пізнішу дату) АБО дійсно означати, що токени не
        # відповідають реальним product-рядкам у корпусі. Розрізнити самостійно: порівняти
        # результат на кількох датах, як у sanity-check.
        print(f"  ℹ сервіс '{service}': 0 CVE у пулі на дату {snapshot_date} для токенів {product_tokens} "
              f"— або жодної відповідної CVE ще не було на цю дату (легітимно), або токени не "
              f"збігаються з products у корпусі (порівняйте з іншою датою, щоб розрізнити)")
    _service_pool_cache[key] = result
    return result

# швидка перевірка на прикладах
for port in (80, 22, 21, 445):
    svc, toks = PORT_SERVICE_MAP[port]
    r = build_service_pool(svc, toks, date(2025, 2, 6))
    cov = f"{r['epss_coverage']:.1%}" if r["epss_coverage"] is not None else "—"
    print(f"port={port:5d} service={svc:6s} n_cve={r['n_cve']:5d} n_epss={r['n_epss']:5d} "
          f"coverage={cov} max_epss={r['max_epss']} in_kev={r['in_kev']} "
          f"generic_os={r['n_generic_os_matches']} specific={r['n_specific_product_matches']}")


port=   80 service=http   n_cve= 1328 n_epss= 1327 coverage=99.9% max_epss=0.97539 in_kev=True generic_os=0 specific=1441
port=   22 service=ssh    n_cve=  114 n_epss=  114 coverage=100.0% max_epss=0.95483 in_kev=False generic_os=0 specific=137
port=   21 service=ftp    n_cve=   71 n_epss=   71 coverage=100.0% max_epss=0.96789 in_kev=False generic_os=0 specific=77
port=  445 service=smb    n_cve= 8221 n_epss= 8212 coverage=99.9% max_epss=0.97505 in_kev=True generic_os=10321 specific=216


In [4]:
# ---------- 3. ПРИЄДНАННЯ КОНТЕКСТУ ДО ДАТАСЕТУ (join по (порт, протокол), не по рядку) ----------
# Колонки з префіксом cve_ctx_, щоб не конфліктувати з наявними полями датасету
# (напр. UNSW-NB15 має власну колонку 'service' з іншим значенням).
#
# НАЗВИ НАВМИСНЕ З ПРЕФІКСОМ 'service_' там, де значення обчислене НА РІВНІ СЕРВІСУ/ПОРТУ,
# а не хоста чи конкретного flow (cve_ctx_service_max_epss, cve_ctx_service_in_kev) —
# щоб унеможливити трактування "цей flow вразливий", коли насправді це "на цьому порту
# в принципі існує клас CVE з таким EPSS/у KEV".
#
# cve_ctx_use_epss — готовий прапорець для downstream ML: True лише якщо EPSS справді є
# в пулі (epss_available) І дата НЕ ретроспективна (тобто EPSS реально існував на snapshot_date,
# а не приписаний заднім числом). Для UNSW-NB15/CIC-IDS2017 це поле буде False майже завжди —
# і це коректно: EPSS для цих датасетів слід використовувати лише як контекстну діагностику,
# ніколи як часову ознаку в leakage-чутливому експерименті.
def attach_cve_context(df, port_col, proto_col, snapshot_date, top_n=20):
    df = df.copy()
    ports = df[port_col].map(parse_port)
    protos = df[proto_col].astype(str).str.lower()

    combo = pd.DataFrame({"__port": ports, "__proto": protos}).drop_duplicates()
    rows = []
    for _, r in combo.iterrows():
        svc, toks = port_to_service(r["__port"], r["__proto"])
        pool = build_service_pool(svc, toks, snapshot_date, top_n)
        use_epss = bool(pool["epss_available"] and not pool["epss_is_retrospective"])
        rows.append(dict(
            __port=r["__port"], __proto=r["__proto"],
            cve_ctx_service=svc,
            cve_ctx_pool_size=pool["n_cve"],
            cve_ctx_context_available=pool["n_cve"] > 0,  # False = немає CVE-мапінгу для цього
                                                           # сервісу (напр. QUIC) — інша ситуація,
                                                           # ніж "мапінг є, але випадково 0 CVE"
            cve_ctx_n_epss=pool["n_epss"],
            cve_ctx_epss_coverage=pool["epss_coverage"],
            cve_ctx_epss_available=pool["epss_available"],
            cve_ctx_epss_is_retrospective=pool["epss_is_retrospective"],
            cve_ctx_use_epss=use_epss,
            cve_ctx_service_max_epss=pool["max_epss"],
            cve_ctx_service_in_kev=pool["in_kev"],
            cve_ctx_n_generic_os_matches=pool["n_generic_os_matches"],
            cve_ctx_n_specific_product_matches=pool["n_specific_product_matches"],
        ))
    lut = pd.DataFrame(rows)

    df["__port"] = ports
    df["__proto"] = protos
    df = df.merge(lut, on=["__port", "__proto"], how="left").drop(columns=["__port", "__proto"])
    assert "cve_ctx_service" in df.columns, "merge не додав очікувані колонки — перевірте на конфлікт назв"
    return df


In [5]:
# ---------- 3.5. SANITY-CHECK: 8 сервісів ПЕРЕД запуском на мільйонах потоків ----------
# Точний список, який найшвидше показує, чи мапінг "порт → продукт" адекватний:
# http, https, ssh, ftp, smb, rdp, quic, rpcbind.
CHECK_COMBOS = [
    (80, "tcp"), (443, "tcp"), (22, "tcp"), (21, "tcp"),
    (445, "tcp"), (3389, "tcp"), (443, "udp"), (111, "tcp"),
]
SANITY_SNAPSHOT = date(2025, 2, 6)   # дата GeNIS — тут EPSS реально існує, зручно для перевірки

rows = []
for port, proto in CHECK_COMBOS:
    svc, toks = port_to_service(port, proto)
    pool = build_service_pool(svc, toks, SANITY_SNAPSHOT, top_n=5)
    rows.append(dict(
        port=port, proto=proto, service=svc, tokens=", ".join(toks) if toks else "—",
        n_cve=pool["n_cve"], n_epss=pool["n_epss"],
        coverage=f"{pool['epss_coverage']:.0%}" if pool["epss_coverage"] is not None else "—",
        max_epss=round(pool["max_epss"], 4) if pool["max_epss"] is not None else None,
        in_kev=pool["in_kev"],
        epss_available=pool["epss_available"], retrospective=pool["epss_is_retrospective"],
        generic_os_matches=pool["n_generic_os_matches"], specific_product_matches=pool["n_specific_product_matches"],
        top5_cve=", ".join(c["cve_id"] for c in pool["top_cve"]),
    ))
sanity_df = pd.DataFrame(rows)
display(sanity_df.drop(columns=["top5_cve"]))
print("\nТоп-5 CVE по кожному сервісу:")
for _, r in sanity_df.iterrows():
    print(f"  {r['port']}/{r['proto']:4s} ({r['service']:8s}): {r['top5_cve'] or '(порожньо)'}")

print(f"\nℹ Числа тут прив'язані до SANITY_SNAPSHOT={SANITY_SNAPSHOT} — для\n"
      f"  UNSW-NB15/CIC-IDS2017 (інші дати) той самий сервіс може дати ІНШЕ n_cve\n"
      f"  через темпоральний фільтр published<=snapshot_date. Це не суперечність,\n"
      f"  а очікувана різниця — деякі CVE ще не існували на ранішу дату.")

zero_services = sanity_df[sanity_df["n_cve"] == 0]["service"].tolist()
if zero_services:
    print(f"\nСервіси з n_cve=0 на цю дату: {zero_services}. Це може бути (а) сервіс навмисне без "
          "надійного CPE-мапінгу (context unavailable — прийнятний, чесний результат, нічого "
          "виправляти не треба), або (б) справді проблема з токенами. Розрізняти за допомогою "
          "cve_ctx_context_available нижче по пайплайну, а не додаючи нові токени навмання.")


,port,proto,service,tokens,n_cve,n_epss,coverage,max_epss,in_kev,epss_available,retrospective,generic_os_matches,specific_product_matches
0,80,tcp,http,"http_server, nginx, iis, php, node.js",1328,1327,100%,0.9754,True,True,False,0,1441
1,443,tcp,https,"http_server, nginx, iis, openssl",700,699,100%,0.9754,True,True,False,0,805
2,22,tcp,ssh,"openssh, dropbear",114,114,100%,0.9548,False,True,False,0,137
3,21,tcp,ftp,"vsftpd, proftpd, filezilla, pure-ftpd, iis",71,71,100%,0.9679,False,True,False,0,77
4,445,tcp,smb,"samba, windows",8221,8212,100%,0.9750,True,True,False,10321,216
5,3389,tcp,rdp,"remote_desktop, remote_desktop_app, remote_desktop_clien...",21,21,100%,0.9574,False,True,False,0,36
6,443,udp,quic,—,0,0,—,NaN,False,False,None,0,0
7,111,tcp,rpcbind,rpcbind,4,4,100%,0.8224,False,True,False,0,4



Топ-5 CVE по кожному сервісу:
  80/tcp  (http    ): CVE-2014-0224, CVE-2015-0235, CVE-2017-9798, CVE-2021-41773, CVE-2010-0425
  443/tcp  (https   ): CVE-2014-0224, CVE-2014-0160, CVE-2014-3566, CVE-2017-9798, CVE-2021-41773
  22/tcp  (ssh     ): CVE-2023-48795, CVE-2006-5051, CVE-2006-4924, CVE-2002-0640, CVE-2003-0693
  21/tcp  (ftp     ): CVE-2015-3306, CVE-2023-48795, CVE-2009-0542, CVE-2019-18217, CVE-2010-4221
  445/tcp  (smb     ): CVE-2015-0318, CVE-2015-0240, CVE-2019-0232, CVE-2015-3090, CVE-2015-3105
  3389/tcp  (rdp     ): CVE-2011-0029, CVE-2013-1296, CVE-2021-38665, CVE-2019-0887, CVE-2021-34535
  443/udp  (quic    ): (порожньо)
  111/tcp  (rpcbind ): CVE-2017-8779, CVE-2015-7236, CVE-2010-2061, CVE-2010-2064

ℹ Числа тут прив'язані до SANITY_SNAPSHOT=2025-02-06 — для
  UNSW-NB15/CIC-IDS2017 (інші дати) той самий сервіс може дати ІНШЕ n_cve
  через темпоральний фільтр published<=snapshot_date. Це не суперечність,
  а очікувана різниця — деякі CVE ще не існували на ранішу

In [6]:
# ---------- 3.6. Сервіси без надійного CPE-мапінгу (фінальний стан, не подальший пошук) ----------
# Це питання ВЖЕ вирішене для RDP (динамічні токени в PORT_SERVICE_MAP) і QUIC (свідомо
# порожній список токенів — 'quic'/'msquic' не є реальними vendor:product у корпусі, і
# шукати заміну через опис CVE чи інші евристики більше НЕ ТРЕБА: чесний
# cve_ctx_context_available=False кращий за штучно притягнутий контекст).
#
# Ця комірка — фінальна перевірка стану, а не інструмент подальшого пошуку токенів:
# перелічує сервіси з довідника, які взагалі не мають товарного мапінгу.
no_mapping = []
for port, entry in PORT_SERVICE_MAP.items():
    variants = entry.items() if isinstance(entry, dict) else [("tcp", entry)]
    for proto, (svc, toks) in variants:
        if not toks:
            no_mapping.append((port, proto, svc))

if no_mapping:
    print("Сервіси без товарного мапінгу (cve_ctx_context_available завжди False для них):")
    for port, proto, svc in no_mapping:
        print(f"  {port}/{proto} → {svc}")
else:
    print("Усі записи довідника мають хоча б один токен продукту.")


Сервіси без товарного мапінгу (cve_ctx_context_available завжди False для них):
  443/udp → quic


In [7]:
# ---------- 4. ЗАСТОСУВАННЯ ДО ТРЬОХ ДАТАСЕТІВ + ДІАГНОСТИКА АСОЦІАЦІЇ КОНТЕКСТУ З МІТКОЮ ----------
# УВАГА: це "CVE-context / label association check", НЕ доказ leakage. Те, що значення
# контексту відрізняються між класами атак, — очікувано (гіпотеза статті саме на це
# розрахована). Захист від ЧАСОВОГО витоку — окрема, вже реалізована темпоральна
# конструкція всередині build_service_pool (published/KEV/EPSS не пізніші за snapshot_date).
DATA = {
    "GeNIS": dict(path="/srv/ids_research/data/genis_flows_10sec.parquet",
                  port_col="Dport_clean", proto_col="Proto", label_col="SubCategoryLabel",
                  snapshot=date(2025, 2, 6)),
    "CIC-IDS2017": dict(path="/srv/ids_research/data/cic_ids2017_flows_fixed.parquet",
                        port_col="destination_port", proto_col="protocol", label_col="attack_label",
                        snapshot=date(2017, 7, 3)),
}
# UNSW-NB15 підвантажимо окремо нижче (сирий CSV, потребує імен колонок)

results = {}
for name, cfg in DATA.items():
    print(f"\n===== {name} =====")
    df = pd.read_parquet(cfg["path"])
    enriched = attach_cve_context(df, cfg["port_col"], cfg["proto_col"], cfg["snapshot"], top_n=20)
    results[name] = enriched

    print("Розподіл за сервісом:")
    print(enriched["cve_ctx_service"].value_counts())

    print(f"\nEPSS для цього датасету придатний як часова ознака (cve_ctx_use_epss): "
          f"{enriched['cve_ctx_use_epss'].mean():.1%} потоків")

    print("\nCVE-context / label association check (НЕ перевірка на витік — витік перевіряється "
          "темпоральною конструкцією всередині build_service_pool, яка вже коректна):")
    agg = enriched.groupby(cfg["label_col"]).agg(
        n=("cve_ctx_service", "size"),
        n_services=("cve_ctx_service", "nunique"),
        mean_service_max_epss=("cve_ctx_service_max_epss", "mean"),
        any_service_in_kev=("cve_ctx_service_in_kev", "mean"),
        share_retrospective_epss=("cve_ctx_epss_is_retrospective", "mean"),  # частка потоків,  # чий EPSS-контекст ретроспективний (не факт, що САМ flow "ретроспективний")
    )
    print(agg)



===== GeNIS =====
Розподіл за сервісом:
cve_ctx_service
http       629110
other      393267
icmp       393216
dns         16452
https       11533
smb         10007
ssh          8770
arp          3775
ftp          3346
quic         2279
netbios       222
mdns          128
smtp           16
imaps          12
Name: count, dtype: int64

EPSS для цього датасету придатний як часова ознака (cve_ctx_use_epss): 46.2% потоків

CVE-context / label association check (НЕ перевірка на витік — витік перевіряється темпоральною конструкцією всередині build_service_pool, яка вже коректна):
                        n  n_services  mean_service_max_epss  any_service_in_kev share_retrospective_epss
SubCategoryLabel                                                                                         
benign-admin         4096           8               0.972023            0.004639                      0.0
benign-background   12862           7               0.973932            0.440833                      

In [8]:
# ---------- 5. UNSW-NB15 (усі 4 файли об'єднано — повний датасет, а не чверть) ----------
UNSW_COLS = ("srcip sport dstip dsport proto state dur sbytes dbytes sttl dttl sloss dloss service Sload Dload Spkts Dpkts "
             "swin dwin stcpb dtcpb smeansz dmeansz trans_depth res_bdy_len Sjit Djit Stime Ltime Sintpkt Dintpkt tcprtt "
             "synack ackdat is_sm_ips_ports ct_state_ttl ct_flw_http_mthd is_ftp_login ct_ftp_cmd ct_srv_src ct_srv_dst "
             "ct_dst_ltm ct_src_ltm ct_src_dport_ltm ct_dst_sport_ltm ct_dst_src_ltm attack_cat Label").split()
assert len(UNSW_COLS) == 49

unsw_parts = []
for i in range(1, 5):
    p = f"/srv/ids_research/data/raw_pcap/UNSW-NB15/UNSW-NB15_{i}.csv"
    df = pd.read_csv(p, names=UNSW_COLS, encoding="utf-8-sig", low_memory=False)
    df["__source_file"] = i
    unsw_parts.append(df)
    print(f"Файл {i}: {len(df):,} рядків")

unsw = pd.concat(unsw_parts, ignore_index=True)
del unsw_parts
unsw["attack_cat"] = unsw["attack_cat"].fillna("Normal").str.strip()
unsw["proto"] = unsw["proto"].astype(str).str.lower()

print(f"\nРазом: {len(unsw):,} потоків")
print(unsw["attack_cat"].value_counts())

# УВАГА: 'Backdoor' і 'Backdoors' НЕ об'єднуються тут автоматично. Перш ніж зливати класи,
# треба перевірити офіційну документацію UNSW-NB15 (Moustafa & Slay), чи це справді один
# клас із варіантом написання, а не навмисно різні категорії. Автоматичне злиття без цієї
# перевірки саме по собі було б методологічною помилкою.
print("\n⚠ Backdoor/Backdoors — окремі класи в цьому прогоні; перевірте документацію UNSW-NB15",
      "перед тим як вирішувати, чи об'єднувати їх:")
print(unsw["attack_cat"].value_counts()[["Backdoor", "Backdoors"]] if "Backdoors" in unsw["attack_cat"].values else "лише один варіант присутній")

enriched_unsw = attach_cve_context(unsw, "dsport", "proto", date(2015, 1, 22), top_n=20)
results["UNSW-NB15"] = enriched_unsw

print("\n===== UNSW-NB15 (повний, 4 файли) =====")
print("Розподіл за сервісом:")
print(enriched_unsw["cve_ctx_service"].value_counts())

print(f"\nEPSS придатний як часова ознака (cve_ctx_use_epss): "
      f"{enriched_unsw['cve_ctx_use_epss'].mean():.1%} потоків "
      f"(очікувано близько до 0% — EPSS не існував у 2015 році)")

print("\nCVE-context / label association check (дата ретроспективна — до появи EPSS,"
      " тому mean_service_max_epss тут НЕ слід трактувати як часову ознаку):")
agg = enriched_unsw.groupby("attack_cat").agg(
    n=("cve_ctx_service", "size"), n_services=("cve_ctx_service", "nunique"),
    mean_service_max_epss=("cve_ctx_service_max_epss", "mean"),
    any_service_in_kev=("cve_ctx_service_in_kev", "mean"),
    share_retrospective_epss=("cve_ctx_epss_is_retrospective", "mean"),
)
print(agg)

# перевірка, що розподіл по кожному з чотирьох файлів окремо узгоджений
print("\nattack_cat по кожному вихідному файлу (перевірка узгодженості):")
print(enriched_unsw.groupby(["__source_file", "attack_cat"]).size().unstack(fill_value=0))


Файл 1: 700,001 рядків
Файл 2: 700,001 рядків
Файл 3: 700,001 рядків
Файл 4: 440,044 рядків

Разом: 2,540,047 потоків
attack_cat
Normal            2218764
Generic            215481
Exploits            44525
Fuzzers             24246
DoS                 16353
Reconnaissance      13987
Analysis             2677
Backdoor             1795
Shellcode            1511
Backdoors             534
Worms                 174
Name: count, dtype: int64

⚠ Backdoor/Backdoors — окремі класи в цьому прогоні; перевірте документацію UNSW-NB15 перед тим як вирішувати, чи об'єднувати їх:
attack_cat
Backdoor     1795
Backdoors     534
Name: count, dtype: int64
  ℹ сервіс 'rpcbind': 0 CVE у пулі на дату 2015-01-22 для токенів ['rpcbind'] — або жодної відповідної CVE ще не було на цю дату (легітимно), або токени не збігаються з products у корпусі (порівняйте з іншою датою, щоб розрізнити)

===== UNSW-NB15 (повний, 4 файли) =====
Розподіл за сервісом:
cve_ctx_service
other         1150148
dns            798968
h

In [9]:
# ---------- 6. ЗБЕРЕЖЕННЯ ЗБАГАЧЕНИХ ДАТАСЕТІВ ----------
# УВАГА: сирі колонки деяких датасетів (напр. UNSW-NB15 'sport'/'dsport') мають ЗМІШАНІ
# типи всередині одного стовпця (частина рядків — Python int, частина — рядки, ймовірно
# через hex-кодування нетипових протоколів в одному з чотирьох вихідних CSV-файлів —
# та сама природа проблеми, що й з Argus-полями Sport/Dport у GeNIS раніше). PyArrow не
# може записати такий стовпець у parquet ("Expected bytes, got a 'int' object"). Це
# проблема сирих даних, не нашого коду збагачення — приводимо всі 'object'-колонки
# до рядкового типу перед записом (як і робили для GeNIS), а не намагаємось вгадати
# "правильний" числовий тип для кожної.
out_dir = Path("/srv/ids_research/data/enriched")
out_dir.mkdir(exist_ok=True)
for name, df in results.items():
    dest = out_dir / f"{name.lower().replace('-', '_')}_with_cve_context.parquet"
    obj_cols = df.select_dtypes(include="object").columns.tolist()
    if obj_cols:
        print(f"{name}: приводжу до рядкового типу колонки зі змішаними типами: {obj_cols}")
        for c in obj_cols:
            df[c] = df[c].astype(str).replace("nan", None)
    df.to_parquet(dest, index=False)
    print(f"Збережено {name}: {dest} ({df.shape})")


GeNIS: приводжу до рядкового типу колонки зі змішаними типами: ['FlowID', 'Proto', 'SrcAddr', 'Sport', 'DstAddr', 'Dport', 'Flgs', 'sDSb', 'dDSb', 'sIpId', 'dIpId', 'Cause', 'Dir', 'State', 'TcpOpt', 'Inode', 'CategoryLabel', 'SubCategoryLabel', '__source_file', 'cve_ctx_service', 'cve_ctx_epss_is_retrospective']


/tmp/ipykernel_16203/2661033109.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include="object").columns.tolist()


Збережено GeNIS: /srv/ids_research/data/enriched/genis_with_cve_context.parquet ((1472133, 140))
CIC-IDS2017: приводжу до рядкового типу колонки зі змішаними типами: ['cve_ctx_service', 'cve_ctx_epss_is_retrospective']


/tmp/ipykernel_16203/2661033109.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include="object").columns.tolist()


Збережено CIC-IDS2017: /srv/ids_research/data/enriched/cic_ids2017_with_cve_context.parquet ((2827677, 96))
UNSW-NB15: приводжу до рядкового типу колонки зі змішаними типами: ['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'service', 'ct_ftp_cmd', 'attack_cat', 'cve_ctx_service', 'cve_ctx_epss_is_retrospective']


/tmp/ipykernel_16203/2661033109.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include="object").columns.tolist()


Збережено UNSW-NB15: /srv/ids_research/data/enriched/unsw_nb15_with_cve_context.parquet ((2540047, 62))


In [10]:
# ---------- 6. ДІАГНОСТИКА ПЕРЕД STEP 4: label → service distribution → KEV ----------
# Мета: перевірити, чи "any_service_in_kev" (і подібні агрегати) несуть інформацію ПОНАД
# сам розподіл сервісів по мітках, чи є повністю похідними від нього.
#
# Логіка: якщо клас атаки майже завжди йде через ОДИН сервіс (вузький розподіл), і цей
# сервіс має in_kev=True, то "label -> KEV" — це просто "label -> port -> KEV" в масці:
# port сам по собі вже (майже) детермінує мітку в цьому датасеті, KEV нічого не додає
# понад те, що вже дає сирий номер порту. Якщо ж розподіл сервісів усередині мітки
# широкий (багато різних портів), а KEV-показник усе одно систематично відрізняється —
# це вже сильніший сигнал, що CVE-контекст додає щось понад голий факт порту.
def label_service_kev_diagnostic(enriched, label_col, top_services=5):
    rows = []
    for label, grp in enriched.groupby(label_col):
        svc_counts = grp["cve_ctx_service"].value_counts()
        top_share = svc_counts.iloc[0] / len(grp) if len(svc_counts) else None   # частка НАЙПОШИРЕНІШОГО сервісу
        dominant_svc = svc_counts.index[0] if len(svc_counts) else None
        rows.append(dict(
            label=label, n=len(grp), n_services=grp["cve_ctx_service"].nunique(),
            dominant_service=dominant_svc, dominant_service_share=top_share,
            any_service_in_kev=grp["cve_ctx_service_in_kev"].mean(),
            top_services=", ".join(f"{s}:{c}" for s, c in svc_counts.head(top_services).items()),
        ))
    out = pd.DataFrame(rows).set_index("label")
    return out.sort_values("dominant_service_share", ascending=False)

print("===== GeNIS: label → service → KEV =====")
diag_genis = label_service_kev_diagnostic(results["GeNIS"], "SubCategoryLabel")
display(diag_genis[["n", "n_services", "dominant_service", "dominant_service_share", "any_service_in_kev"]])
print("\nІнтерпретація: dominant_service_share ≈ 1.0 означає, що мітка практично ЗАВЖДИ")
print("йде через один сервіс/порт — тоді будь-яка асоціація цього класу з KEV є похідною")
print("від порту, а не незалежним сигналом. dominant_service_share << 1.0 при цьому")
print("систематичному any_service_in_kev — сильніший доказ, що контекст щось додає.")

print("\n\n===== CIC-IDS2017: label → service → KEV =====")
diag_cic = label_service_kev_diagnostic(results["CIC-IDS2017"], "attack_label")
display(diag_cic[["n", "n_services", "dominant_service", "dominant_service_share", "any_service_in_kev"]])

print("\n\n===== UNSW-NB15: label → service → KEV =====")
diag_unsw = label_service_kev_diagnostic(enriched_unsw, "attack_cat")
display(diag_unsw[["n", "n_services", "dominant_service", "dominant_service_share", "any_service_in_kev"]])


===== GeNIS: label → service → KEV =====


,n,n_services,dominant_service,dominant_service_share,any_service_in_kev
label,,,,,
bruteforce-ssh,8677,1,ssh,1.000000,0.000000
bruteforce-smb,10001,1,smb,1.000000,1.000000
bruteforce-ftp,3344,1,ftp,1.000000,0.000000
dos-icmp,393216,1,icmp,1.000000,0.000000
dos-pushack,392736,1,http,1.000000,1.000000
dos-slowloris,168116,1,http,1.000000,1.000000
dos-hulk,67821,1,http,1.000000,1.000000
dos-udp,393216,1,other,1.000000,0.000000
benign-admin,4096,8,dns,0.869629,0.004639



Інтерпретація: dominant_service_share ≈ 1.0 означає, що мітка практично ЗАВЖДИ
йде через один сервіс/порт — тоді будь-яка асоціація цього класу з KEV є похідною
від порту, а не незалежним сигналом. dominant_service_share << 1.0 при цьому
систематичному any_service_in_kev — сильніший доказ, що контекст щось додає.


===== CIC-IDS2017: label → service → KEV =====


,n,n_services,dominant_service,dominant_service_share,any_service_in_kev
label,,,,,
DoS Slowhttptest,5499,1,http,1.000000,0.0
DoS Hulk,230123,1,http,1.000000,0.0
DoS GoldenEye,10293,1,http,1.000000,0.0
DoS slowloris,5796,1,http,1.000000,0.0
Web Attack – Brute Force,1507,1,http,1.000000,0.0
Infiltration,36,1,other,1.000000,0.0
Heartbleed,11,1,other,1.000000,0.0
Web Attack – Sql Injection,21,1,http,1.000000,0.0
Web Attack – XSS,652,1,http,1.000000,0.0




===== UNSW-NB15: label → service → KEV =====


,n,n_services,dominant_service,dominant_service_share,any_service_in_kev
label,,,,,
Shellcode,1511,1,other,1.000000,0.0
Backdoors,534,5,other,0.975655,0.0
Generic,215481,14,dns,0.973677,0.0
Backdoor,1795,5,other,0.954875,0.0
Worms,174,2,http,0.879310,0.0
DoS,16353,19,other,0.818993,0.0
Analysis,2677,3,other,0.767650,0.0
Reconnaissance,13987,8,rpcbind,0.680918,0.0
Fuzzers,24246,7,other,0.588509,0.0
